---
## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objectives & Concepts
- **Template Method Pattern:** The base class defines the skeleton of the algorithm (`run()`), and subclasses implement the individual steps.
- **Composition:** The pipeline reuses `CSVReader`, `DataCleaner` and the `DataTransform` objects from the previous exercises.
- **Abstraction:** `@abstractmethod` forces every pipeline to implement `load`, `clean`, `transform` and `save`.

### Instructions:
1. Create an abstract class `DataPipeline` with a `run()` method that calls: `load()`, `clean()`, `transform()`, `save()`.
2. Create `CSVDataPipeline` that implements each step using the previous classes.

In [5]:
from abc import ABC, abstractmethod

In [6]:
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np

# ==========================================
# 1. CORE PIPELINE FRAMEWORK (ABCs)
# ==========================================

class DataPipeline(ABC):
    def run(self):                      # fixed order of steps
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        return df

    @abstractmethod
    def load(self): pass
    @abstractmethod
    def clean(self, df): pass
    @abstractmethod
    def transform(self, df): pass
    @abstractmethod
    def save(self, df): pass


# ==========================================
# 2. IO COMPONENTS (Reader)
# ==========================================

class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        
    def read(self):
        # Reads the CSV file into a pandas DataFrame
        return pd.read_csv(self.file_path)


# ==========================================
# 3. CLEANING COMPONENTS (Strategy Pattern)
# ==========================================

class CleaningStrategy(ABC):
    @abstractmethod
    def apply(self, df): pass

class FillMean(CleaningStrategy):
    def apply(self, df):
        # Fills missing numeric values with the column mean
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
        return df

class DataCleaner:
    def __init__(self, strategy: CleaningStrategy):
        self.strategy = strategy
        
    def clean(self, df):
        # Delegates cleaning to the selected strategy
        return self.strategy.apply(df)


# ==========================================
# 4. TRANSFORMATION COMPONENTS (Factory Pattern)
# ==========================================

class Transform(ABC):
    @abstractmethod
    def apply(self, df): pass

class RemoveDuplicates(Transform):
    def apply(self, df):
        return df.drop_duplicates()

class StandardizeText(Transform):
    def apply(self, df):
        # Converts all text columns to lowercase and strips whitespace
        text_cols = df.select_dtypes(include=['object']).columns
        for col in text_cols:
            df[col] = df[col].astype(str).str.lower().str.strip()
        return df

class Normalize(Transform):
    def apply(self, df):
        # Min-Max Normalization for numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df[col].max() != df[col].min():
                df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
        return df

class TransformFactory:
    @staticmethod
    def get_transform(transform_type: str) -> Transform:
        transforms = {
            "remove_duplicates": RemoveDuplicates(),
            "standardize_text": StandardizeText(),
            "normalize": Normalize()
        }
        if transform_type in transforms:
            return transforms[transform_type]
        raise ValueError(f"Unknown transform type: {transform_type}")


# ==========================================
# 5. CONCRETE PIPELINE IMPLEMENTATION
# ==========================================

class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, cleaner, transforms):
        self.reader = CSVReader(input_path)
        self.output_path = output_path
        self.cleaner = cleaner
        self.transforms = transforms

    def load(self):
        return self.reader.read()

    def clean(self, df):
        return self.cleaner.clean(df)

    def transform(self, df):
        for t in self.transforms:
            df = t.apply(df)
        return df

    def save(self, df):
        df.to_csv(self.output_path, index=False)


# ==========================================
# 6. EXECUTION ENVIRONMENT (Testing the Setup)
# ==========================================

if __name__ == "__main__":
    # Create a dummy 'sample_data.csv' file for verification
    mock_data = {
        'Name': [' Alice ', 'Bob', 'Alice ', 'Charlie'],
        'Score': [90.0, np.nan, 90.0, 70.0],
        'Age': [25, 30, 25, 22]
    }
    pd.DataFrame(mock_data).to_csv("sample_data.csv", index=False)

    # Initialize and execute your pipeline setup
    pipeline = CSVDataPipeline(
        "sample_data.csv",
        "cleaned_sample_data.csv",
        DataCleaner(FillMean()),
        [TransformFactory.get_transform("remove_duplicates"),
         TransformFactory.get_transform("standardize_text"),
         TransformFactory.get_transform("normalize")],
    )
    
    result = pipeline.run()
    print(result.round(2))


      Name  Score   Age
0    alice   1.00  0.38
1      bob   0.67  1.00
2    alice   1.00  0.38
3  charlie   0.00  0.00


C:\Users\DELL\AppData\Local\Temp\ipykernel_24548\1125907293.py:79: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=['object']).columns
